# Notebook: Qualitative Evaluation (Expert Analysis) -- Experiment 1

In this notebook, the qualitative evaluation of the first experiment is conducted, comparing the ratings of supervisors and staff members.

The first evaluation run in this notebook is based on the data from the supervisor (expert 1), while `USE_SUPERVISOR_DATA` is set to `True`.

If set to `False`, the notebook will use the data from the staff members (experts 2-5) instead as the second run. Furthermore, if set to `False`, the inter-rater agreement (IRA) will be calculated between the staff members + between the staff members and the supervisor.

## Initial Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
from statsmodels.stats.inter_rater import fleiss_kappa

warnings.filterwarnings('ignore')

# Plot configuration
plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# Display configuration
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.notebook_repr_html', True)

# !!! Set to True to use ONLY supervisor data (Expert 1 - Professor)
# !!! Set to False to use ALL 5 experts data (Experts 1-5 combined, with inter-rater agreement analysis)
USE_SUPERVISOR_DATA = True

# Path configuration
BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
qualitative_base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/qualitative/exp1")
sampled_hints_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/sampled/exp1_sampled.csv")
output_base_path = os.path.join(BASE_PROJECT_PATH, "40_evaluation/exp1/qualitative")
output_tables_path = os.path.join(output_base_path, "tables")
output_plots_path = os.path.join(output_base_path, "plots")

for path in [output_tables_path, output_plots_path]:
    os.makedirs(path, exist_ok=True)

# Global Label Mappings - used throughout the notebook for consistent labeling
LABEL_MAPPING = {
    'anthropic': 'Anthropic',
    'openai': 'OpenAI',
    'deepseek': 'DeepSeek',
    'xai': 'xAI',
    'google': 'Google',
    'mcq': 'MCQ',
    'open_ended': 'Open-Ended',
    'layer1': 'Layer 1',
    'layer2': 'Layer 2',
    'layer3': 'Layer 3',
    'layer4': 'Layer 4',
    'layer5': 'Layer 5',
    'layer6': 'Layer 6',
    'layer7': 'Layer 7'
}

# Storage for results
tables = {}
plots = {}

print("Setup completed successfully")
print(f"Output tables: {output_tables_path}")
print(f"Output plots: {output_plots_path}")
print(f"Data source: {'ONLY SUPERVISOR (Expert 1)' if USE_SUPERVISOR_DATA else 'ALL 5 EXPERTS (Experts 1-5)'}")


In [ ]:
# Utility Functions

def create_seaborn_boxplot(data, x, y, ax, title, ylabel, xlabel, scale_range=None):
    colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
    unique_vals = sorted(data[x].unique())
    palette = colors[:len(unique_vals)]
    
    sns.boxplot(data=data, x=x, y=y, ax=ax, palette=palette,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    for i, val in enumerate(unique_vals):
        mean_val = data[data[x] == val][y].mean()
        ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, 
                  edgecolor='darkred', linewidth=1)
    
    # Use global LABEL_MAPPING
    current_labels = [tick.get_text() for tick in ax.get_xticklabels()]
    new_labels = [LABEL_MAPPING.get(label, label) for label in current_labels]
    ax.set_xticklabels(new_labels, rotation=0)
    
    if scale_range:
        ax.set_ylim(scale_range)
        if scale_range == (0, 10):
            title += " (0-10 scale)"
        elif scale_range == (0, 70):
            title += " (0-70 scale)"
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=15)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.grid(True, alpha=0.3)


In [ ]:
def load_supervisor_data():
    expert1_file = os.path.join(qualitative_base_path, "exp1_eval_e1.csv")
    
    if not os.path.exists(expert1_file):
        raise FileNotFoundError(f"Expert 1 data not found at {expert1_file}")
    
    expert1_df = pd.read_csv(expert1_file)
    
    if os.path.exists(sampled_hints_path):
        sampled_df = pd.read_csv(sampled_hints_path)
        
        sampled_df['sample_id'] = range(1, len(sampled_df) + 1)
        
        if 'sample_id' in expert1_df.columns:
            expert1_df['sample_id'] = expert1_df['sample_id'].astype(int)
            
            exp1_df = pd.merge(expert1_df, sampled_df[['sample_id', 'llm', 'bloom_idx']], 
                              on='sample_id', how='left', suffixes=('', '_sampled'))
            
            duplicate_cols = [col for col in exp1_df.columns if col.endswith('_sampled')]
            if duplicate_cols:
                exp1_df = exp1_df.drop(columns=duplicate_cols)
        else:
            print("Warning: No sample_id in expert1_df")
            exp1_df = expert1_df
    else:
        exp1_df = expert1_df
        print(f"Warning: Sampled file not found at {sampled_hints_path}")
    
    if 'layer' in exp1_df.columns:
        exp1_df['layer'] = pd.to_numeric(exp1_df['layer'], errors='coerce')
        exp1_df['layer'] = exp1_df['layer'].astype('Int64')
    
    if 'question_type' in exp1_df.columns:
        valid_types = ['mcq', 'open_ended']
        exp1_df['question_type'] = exp1_df['question_type'].apply(
            lambda x: x if pd.notna(x) and x in valid_types else pd.NA
        )
    
    return exp1_df


In [ ]:
def load_all_experts_data():
    if not os.path.exists(sampled_hints_path):
        raise FileNotFoundError(f"Sampled file not found at {sampled_hints_path}")
    
    sampled_df = pd.read_csv(sampled_hints_path)
    sampled_df['sample_id'] = range(1, len(sampled_df) + 1)
    
    experts_data = {}
    all_expert_dfs = []
    
    for expert_num in [1, 2, 3, 4, 5]:
        expert_key = f'expert_{expert_num}'
        expert_file = os.path.join(qualitative_base_path, f"exp1_eval_e{expert_num}.csv")
        
        if not os.path.exists(expert_file):
            print(f"Warning: {expert_file} not found, skipping Expert {expert_num}")
            continue
        
        expert_df = pd.read_csv(expert_file)
        
        if 'sample_id' in expert_df.columns:
            expert_df['sample_id'] = expert_df['sample_id'].astype(int)
            expert_df = pd.merge(expert_df, sampled_df[['sample_id', 'llm', 'bloom_idx']], 
                                on='sample_id', how='left', suffixes=('', '_sampled'))
            
            duplicate_cols = [col for col in expert_df.columns if col.endswith('_sampled')]
            if duplicate_cols:
                expert_df = expert_df.drop(columns=duplicate_cols)
        
        if 'layer' in expert_df.columns:
            expert_df['layer'] = pd.to_numeric(expert_df['layer'], errors='coerce')
            expert_df['layer'] = expert_df['layer'].astype('Int64')
        
        if 'question_type' in expert_df.columns:
            valid_types = ['mcq', 'open_ended']
            expert_df['question_type'] = expert_df['question_type'].apply(
                lambda x: x if pd.notna(x) and x in valid_types else pd.NA
            )
        
        expert_df['expert'] = expert_key
        experts_data[expert_key] = expert_df.copy()
        all_expert_dfs.append(expert_df)
    
    if all_expert_dfs:
        exp1_df = pd.concat(all_expert_dfs, ignore_index=True)
    else:
        exp1_df = pd.DataFrame()
    
    return exp1_df, experts_data


In [ ]:
def clean_numeric_data(df, numeric_cols):
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)
            invalid_values = ['??', '???', '?', '????', '', ' ', 'nan', 'NaN', 'NULL', 'null', 

                             'None', 'NONE', 'n/a', 'N/A', '#N/A', '#NULL!', 
                             'undefined', 'UNDEFINED', '-', '--', '---']

            df[col] = df[col].replace(invalid_values, np.nan)
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
            valid_data = df[col].dropna()
            if len(valid_data) > 0:
                out_of_range = valid_data[(valid_data < 0) | (valid_data > 10)]
                if len(out_of_range) > 0:
                    df.loc[(df[col] < 0) | (df[col] > 10), col] = np.nan
    return df

In [ ]:
def calculate_fleiss_kappa(expert_dfs_dict, criteria_cols):
    results = []
    
    expert_keys = sorted(expert_dfs_dict.keys())
    
    if len(expert_keys) < 2:
        print("Need at least 2 experts for agreement analysis")
        return pd.DataFrame()
    
    common_ids = None
    for expert_key in expert_keys:
        df = expert_dfs_dict[expert_key]
        if 'sample_id' in df.columns:
            ids = set(df['sample_id'].astype(int))
            if common_ids is None:
                common_ids = ids
            else:
                common_ids = common_ids & ids
    
    if not common_ids:
        print("No common sample_ids found across experts")
        return pd.DataFrame()
    
    common_ids = sorted(list(common_ids))
    print(f"Analyzing {len(expert_keys)} experts with {len(common_ids)} common samples")
    
    for criterion in criteria_cols:
        ratings = []
        for sample_id in common_ids:
            sample_ratings = []
            valid_sample = True
            
            for expert_key in expert_keys:
                df = expert_dfs_dict[expert_key]
                if sample_id in df['sample_id'].values:
                    rating = df[df['sample_id'] == sample_id][criterion].iloc[0]
                    
                    def safe_convert(val):
                        return np.nan if pd.isna(val) else float(val)
                    
                    rating_val = safe_convert(rating)
                    
                    if pd.isna(rating_val) or rating_val < 0 or rating_val > 10:
                        valid_sample = False
                        break
                    sample_ratings.append(int(rating_val))
                else:
                    valid_sample = False
                    break
            
            if valid_sample and len(sample_ratings) == len(expert_keys):
                ratings.append(sample_ratings)
        
        if len(ratings) >= 2:
            ratings_array = np.array(ratings)
            fleiss_table = np.zeros((len(ratings), 11))
            
            for i, item_ratings in enumerate(ratings):
                for rating in item_ratings:
                    fleiss_table[i, int(rating)] += 1
            
            kappa = fleiss_kappa(fleiss_table)
            
            level = ("Slight" if kappa < 0.2 else 
                    "Fair" if kappa < 0.4 else 
                    "Moderate" if kappa < 0.6 else 
                    "Substantial" if kappa < 0.8 else 
                    "Almost Perfect")
            
            results.append({
                'Criterion': criterion,
                'Fleiss_Kappa': round(kappa, 3),
                'Agreement_Level': level,
                'N_Items': len(ratings),
                'N_Raters': len(expert_keys),
                'Mean_Rating': round(np.mean(ratings_array), 2),
                'Std_Rating': round(np.std(ratings_array), 2)
            })
    
    return pd.DataFrame(results)

print("Utility functions loaded successfully")

## Data Loading and Configuration

In [ ]:
# Load data based on configuration
if USE_SUPERVISOR_DATA:
    print("Loading SUPERVISOR data ONLY (Expert 1 - single rater)")
    exp1_df = load_supervisor_data()
    analysis_suffix = "supervisor"
    agreement_available = False
else:
    print("Loading ALL EXPERTS data (Experts 1-5 - multiple raters with agreement analysis)")
    exp1_df, experts_data = load_all_experts_data()
    analysis_suffix = "all_experts" 
    agreement_available = True

numeric_cols = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'correctness']

exp1_df = clean_numeric_data(exp1_df, numeric_cols)
exp1_df['total_score'] = exp1_df[numeric_cols].sum(axis=1)

exp1_filled = exp1_df[numeric_cols].notna().sum().sum()
exp1_total = len(exp1_df) * len(numeric_cols)

print(f"\nData loaded successfully!")
print(f"Experiment 1: {len(exp1_df)} samples, {100*exp1_filled/exp1_total:.1f}% completion")
print(f"Analysis suffix: {analysis_suffix}")

if 'llm' in exp1_df.columns:
    print(f"\nDistribution:")
    print(f"  LLMs: {list(exp1_df['llm'].unique())}")
if 'question_type' in exp1_df.columns:
    print(f"  Question Types: {list(exp1_df['question_type'].unique())}")
if 'layer' in exp1_df.columns:
    print(f"  Layers (OSI): {sorted(exp1_df['layer'].unique())}")

if 'layer' in exp1_df.columns:
    if 'input_source' not in exp1_df.columns:
        exp1_df['input_source'] = 'layer' + exp1_df['layer'].astype(str)
if 'question_type' in exp1_df.columns:
    if 'prompt_type' not in exp1_df.columns:
        exp1_df['prompt_type'] = exp1_df['question_type']

# Experiment 1: OSI Layer-Based Analysis

Analysis of LLM question generation quality using OSI layer source materials.

In [ ]:
print("EXPERIMENT 1 - DESCRIPTIVE STATISTICS")
print("="*60)

criteria = numeric_cols + ['total_score']

# Overall statistics
exp1_stats = exp1_df[numeric_cols].describe().round(2)
tables[f'exp1_overall_stats_{analysis_suffix}'] = exp1_stats
print("\nOverall Statistics:")
display(exp1_stats)

In [ ]:
# Statistics by LLM
exp1_llm_stats = exp1_df.groupby('llm')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp1_llm_stats_{analysis_suffix}'] = exp1_llm_stats
print("\nStatistics by LLM:")
display(exp1_llm_stats)

In [ ]:
# LLM ranking
exp1_llm_means = exp1_df.groupby('llm')[numeric_cols].mean().round(2)
exp1_llm_overall = exp1_llm_means.mean(axis=1).sort_values(ascending=False)
tables[f'exp1_llm_ranking_{analysis_suffix}'] = exp1_llm_overall

print("\nOverall LLM Ranking:")
for i, (llm, score) in enumerate(exp1_llm_overall.items(), 1):
    print(f"{i}. {llm.title()}: {score:.2f}")

In [ ]:
# Statistics by Input Source (OSI Layer)
exp1_source_stats = exp1_df.groupby('input_source')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp1_source_stats_{analysis_suffix}'] = exp1_source_stats
print("\nStatistics by Input Source (OSI Layer):")
display(exp1_source_stats)

In [ ]:
# Statistics by Prompt Type (Question Type)
exp1_prompt_stats = exp1_df.groupby('prompt_type')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp1_prompt_stats_{analysis_suffix}'] = exp1_prompt_stats
print("\nStatistics by Prompt Type (Question Type):")
display(exp1_prompt_stats)

In [ ]:
# LLM Performance Visualization
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()
plots[f'exp1_llm_analysis_{analysis_suffix}'] = fig

for i, criterion in enumerate(numeric_cols):
    create_seaborn_boxplot(exp1_df, 'llm', criterion, axes[i], 
                          f'{criterion.title()}', criterion.title(), 'LLM', scale_range=(0, 10))

create_seaborn_boxplot(exp1_df, 'llm', 'total_score', axes[7], 
                      'Total Score', 'Total Score', 'LLM', scale_range=(0, 70))

plt.suptitle('Experiment 1: LLM Performance across all Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

In [ ]:
# Input Source Analysis (OSI Layers)
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()
plots[f'exp1_source_analysis_{analysis_suffix}'] = fig

for i, criterion in enumerate(numeric_cols):
    create_seaborn_boxplot(exp1_df, 'input_source', criterion, axes[i], 
                          f'{criterion.title()}', criterion.title(), 'OSI Layer', scale_range=(0, 10))

create_seaborn_boxplot(exp1_df, 'input_source', 'total_score', axes[7], 
                      'Total Score', 'Total Score', 'OSI Layer', scale_range=(0, 70))

plt.suptitle('Experiment 1: Performance by OSI Layer', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

In [ ]:
# Question Type Analysis
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
plots[f'exp1_prompt_analysis_{analysis_suffix}'] = fig

for i, criterion in enumerate(numeric_cols):
    create_seaborn_boxplot(exp1_df, 'prompt_type', criterion, axes[i], 
                          f'{criterion.title()}', criterion.title(), 'Question Type', scale_range=(0, 10))

create_seaborn_boxplot(exp1_df, 'prompt_type', 'total_score', axes[7], 
                      'Total Score', 'Total Score', 'Question Type', scale_range=(0, 70))

plt.suptitle('Experiment 1: Performance by Question Type', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
# Total Score Heatmap: LLM vs Question Type
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plots[f'exp1_total_score_heatmap_{analysis_suffix}'] = fig

# Total Score by LLM vs Question Type
heatmap_mean_prompt = exp1_df.groupby(['llm', 'prompt_type'])['total_score'].mean().unstack()
heatmap_std_prompt = exp1_df.groupby(['llm', 'prompt_type'])['total_score'].std().unstack()

annot_matrix_prompt = heatmap_mean_prompt.copy()
for i in range(len(heatmap_mean_prompt.index)):
    for j in range(len(heatmap_mean_prompt.columns)):
        mean_val = heatmap_mean_prompt.iloc[i, j]
        std_val = heatmap_std_prompt.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_prompt.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            annot_matrix_prompt.iloc[i, j] = f"{mean_val:.1f}"

# Use global LABEL_MAPPING
llm_labels = [LABEL_MAPPING.get(label, label.title()) for label in heatmap_mean_prompt.index]
prompt_labels = [LABEL_MAPPING.get(label, label.title()) for label in heatmap_mean_prompt.columns]

sns.heatmap(heatmap_mean_prompt, annot=annot_matrix_prompt, fmt='', cmap='RdYlBu_r',
            center=heatmap_mean_prompt.mean().mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Total Score'}, annot_kws={'size': 11, 'weight': 'bold'},
            xticklabels=prompt_labels, yticklabels=llm_labels, ax=ax)

ax.set_title('Total Score: LLM vs Question Type\nValues: Mean (Std) | Scale: 0-70 points', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Question Type', fontsize=12, fontweight='bold')
ax.set_ylabel('LLM', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Correctness Heatmap: LLM vs Question Type
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plots[f'exp1_correctness_heatmap_{analysis_suffix}'] = fig

# Correctness by LLM vs Question Type
correctness_prompt_heatmap_mean = exp1_df.groupby(['llm', 'prompt_type'])['correctness'].mean().unstack()
correctness_prompt_heatmap_std = exp1_df.groupby(['llm', 'prompt_type'])['correctness'].std().unstack()

correctness_prompt_annot_matrix = correctness_prompt_heatmap_mean.copy()
for i in range(len(correctness_prompt_heatmap_mean.index)):
    for j in range(len(correctness_prompt_heatmap_mean.columns)):
        mean_val = correctness_prompt_heatmap_mean.iloc[i, j]
        std_val = correctness_prompt_heatmap_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            correctness_prompt_annot_matrix.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            correctness_prompt_annot_matrix.iloc[i, j] = f"{mean_val:.1f}"

# Use global LABEL_MAPPING
llm_labels = [LABEL_MAPPING.get(l, l.title()) for l in correctness_prompt_heatmap_mean.index]
prompt_labels = [LABEL_MAPPING.get(l, l.title()) for l in correctness_prompt_heatmap_mean.columns]

sns.heatmap(correctness_prompt_heatmap_mean, annot=correctness_prompt_annot_matrix, fmt='', cmap='RdYlBu_r',
            center=correctness_prompt_heatmap_mean.mean().mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correctness Score'}, annot_kws={'size': 11, 'weight': 'bold'},
            xticklabels=prompt_labels, yticklabels=llm_labels, ax=ax)

ax.set_title('Correctness Score: LLM vs Question Type\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Question Type', fontsize=12, fontweight='bold')
ax.set_ylabel('LLM', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Store detailed statistics tables
exp1_llm_source_combinations = exp1_df.groupby(['llm', 'input_source'])['total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_llm_source_combinations_{analysis_suffix}'] = exp1_llm_source_combinations

exp1_llm_prompt_combinations = exp1_df.groupby(['llm', 'prompt_type'])['total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_llm_prompt_combinations_{analysis_suffix}'] = exp1_llm_prompt_combinations

correctness_llm_source_stats = exp1_df.groupby(['llm', 'input_source'])['correctness'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_correctness_llm_source_stats_{analysis_suffix}'] = correctness_llm_source_stats

correctness_llm_prompt_stats = exp1_df.groupby(['llm', 'prompt_type'])['correctness'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_correctness_llm_prompt_stats_{analysis_suffix}'] = correctness_llm_prompt_stats


In [ ]:
# Correctness Heatmap: LLM vs Question Type
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plots[f'exp1_correctness_heatmap_{analysis_suffix}'] = fig

# Correctness by LLM vs Question Type
correctness_prompt_heatmap_mean = exp1_df.groupby(['llm', 'prompt_type'])['correctness'].mean().unstack()
correctness_prompt_heatmap_std = exp1_df.groupby(['llm', 'prompt_type'])['correctness'].std().unstack()

correctness_prompt_annot_matrix = correctness_prompt_heatmap_mean.copy()
for i in range(len(correctness_prompt_heatmap_mean.index)):
    for j in range(len(correctness_prompt_heatmap_mean.columns)):
        mean_val = correctness_prompt_heatmap_mean.iloc[i, j]
        std_val = correctness_prompt_heatmap_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            correctness_prompt_annot_matrix.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            correctness_prompt_annot_matrix.iloc[i, j] = f"{mean_val:.1f}"

# Label mapping
llm_label_map = {'anthropic': 'Anthropic', 'openai': 'OpenAI', 'deepseek': 'DeepSeek', 'xai': 'xAI', 'google': 'Google'}
prompt_label_map = {'mcq': 'MCQ', 'open_ended': 'Open-Ended'}

llm_labels = [llm_label_map.get(l, l.title()) for l in correctness_prompt_heatmap_mean.index]
prompt_labels = [prompt_label_map.get(l, l.title()) for l in correctness_prompt_heatmap_mean.columns]

sns.heatmap(correctness_prompt_heatmap_mean, annot=correctness_prompt_annot_matrix, fmt='', cmap='RdYlBu_r',
            center=correctness_prompt_heatmap_mean.mean().mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correctness Score'}, annot_kws={'size': 11, 'weight': 'bold'},
            xticklabels=prompt_labels, yticklabels=llm_labels, ax=ax)

ax.set_title('Correctness Score: LLM vs Question Type\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Question Type', fontsize=12, fontweight='bold')
ax.set_ylabel('LLM', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Store detailed statistics tables
exp1_llm_source_combinations = exp1_df.groupby(['llm', 'input_source'])['total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_llm_source_combinations_{analysis_suffix}'] = exp1_llm_source_combinations

exp1_llm_prompt_combinations = exp1_df.groupby(['llm', 'prompt_type'])['total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_llm_prompt_combinations_{analysis_suffix}'] = exp1_llm_prompt_combinations

correctness_llm_source_stats = exp1_df.groupby(['llm', 'input_source'])['correctness'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_correctness_llm_source_stats_{analysis_suffix}'] = correctness_llm_source_stats

correctness_llm_prompt_stats = exp1_df.groupby(['llm', 'prompt_type'])['correctness'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_correctness_llm_prompt_stats_{analysis_suffix}'] = correctness_llm_prompt_stats


# Inter-Rater Agreement Analysis

**Note:** Agreement analysis is only available when all 5 experts data is loaded (USE_SUPERVISOR_DATA = False).

In [ ]:
if agreement_available: # if using all experts data
    print("INTER-RATER AGREEMENT ANALYSIS (Fleiss' Kappa)")
    print("="*60)
    print(f"Analyzing agreement across ALL 5 EXPERTS (Experts 1-5)")
    
    # Experiment 1 Agreement - ALL 5 EXPERTS
    print("\nExperiment 1 Agreement (All 5 Experts):")
    
    # Prepare expert dataframes dictionary
    expert_dfs_dict = {}
    for expert_key in ['expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']:
        if expert_key in experts_data:
            expert_dfs_dict[expert_key] = experts_data[expert_key]
    
    agreement_exp1 = calculate_fleiss_kappa(expert_dfs_dict, numeric_cols)
    
    if not agreement_exp1.empty:
        tables[f'agreement_exp1_{analysis_suffix}'] = agreement_exp1
        display(agreement_exp1.round(3))
        
        valid_kappas = agreement_exp1['Fleiss_Kappa'].dropna()
        if len(valid_kappas) > 0:
            avg_kappa = valid_kappas.mean()
            
            level = ("Slight" if avg_kappa < 0.2 else 
                    "Fair" if avg_kappa < 0.4 else 
                    "Moderate" if avg_kappa < 0.6 else 
                    "Substantial" if avg_kappa < 0.8 else 
                    "Almost Perfect")
            
            print(f"\nAverage Fleiss' Kappa: {avg_kappa:.3f} ({level})")
    
    # Expert Comparison - Show individual expert statistics
    print("\n" + "="*60)
    print("EXPERT COMPARISON - INDIVIDUAL STATISTICS")
    print("="*60)
    
    expert_means = {}
    for expert_key in ['expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']:
        if expert_key in experts_data:
            expert_means[expert_key] = experts_data[expert_key][numeric_cols].mean()
    
    if expert_means:
        comparison_df = pd.DataFrame(expert_means).T
        comparison_df['Overall_Mean'] = comparison_df.mean(axis=1)
        comparison_df = comparison_df.round(3)
        tables[f'expert_comparison_exp1_{analysis_suffix}'] = comparison_df
        
        print("\nMean Ratings by Expert (Experiment 1):")
        display(comparison_df)
        
else:
    print("INTER-RATER AGREEMENT ANALYSIS")
    print("="*60)
    print("Agreement analysis not available for supervisor-only data.")
    print("Switch to all experts mode (USE_SUPERVISOR_DATA = False) to enable agreement analysis.")

In [ ]:
# Additional detailed agreement analysis if needed
if agreement_available:
    print("DETAILED AGREEMENT ANALYSIS")
    print("="*70)
    
    # You can add more detailed analyses here if needed
    # For example: pairwise comparisons, correlation matrices, etc.
    
    print("\nNote: This cell can be expanded for additional agreement analyses")
    print("Current analysis shows Fleiss' Kappa for all 5 experts combined")
    print("Based on OSI layer data without manipulations")

## Data Export

Save all tables and plots for thesis and presentations.

In [ ]:
def save_all_results():
    print("Saving results...")
    
    # Determine prefix based on data source
    prefix = "supervisor_" if USE_SUPERVISOR_DATA else "all_experts_"
    
    tables_saved = 0
    for table_name, table_data in tables.items():
        clean_name = table_name.replace(f"_{analysis_suffix}", "")
        csv_path = os.path.join(output_tables_path, f"{prefix}{clean_name}.csv")
        table_data.to_csv(csv_path)
        tables_saved += 1
        print(f"Saved table: {prefix}{clean_name}.csv")
    
    plots_saved = 0
    for plot_name, plot_fig in plots.items():
        clean_name = plot_name.replace(f"_{analysis_suffix}", "")
        png_path = os.path.join(output_plots_path, f"{prefix}{clean_name}.png")
        plot_fig.savefig(png_path, dpi=300, bbox_inches='tight')
        plots_saved += 1
        print(f"Saved plot: {prefix}{clean_name}.png")
    
    print(f"\nExport Summary:")
    print(f"  Tables saved: {tables_saved}")
    print(f"  Plots saved: {plots_saved}")
    print(f"  Output location: {output_base_path}")
    print(f"  Analysis type: {analysis_suffix}")
    print(f"  File prefix: {prefix}")
    print(f"  Data source: {'Only Expert 1 (Supervisor)' if USE_SUPERVISOR_DATA else 'All 5 Experts (1-5)'}")

save_all_results()